# GeoLift-RT v2.1 — inspect fused geometry trước, rồi final train

Notebook này cố ý tách pipeline thành hai cổng kiểm tra:

1. **Inspect gate:** stream vài NPZ từ `geometry_fused_train.tar`, kiểm tra key/schema; sau khi extract thì kiểm tra coverage và hiển thị riêng train/val.
2. **Final-train gate:** chỉ train nếu metric, DA raw và fused geometry cùng khớp 100% sample ID, train/val không trùng ID/raw drive, và loader đọc đúng `R_G/C_G`.

Teacher dùng trong loss cuối: metric `D_cm/C_cm` và fused geometry `R_G/C_G`. DA raw được giữ để inspect/audit; không được dùng làm fallback nếu fused geometry thiếu.

In [ ]:
# 1) Cấu hình
from pathlib import Path
REPO_GIT_URL = ''
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/GeoDistill_RT')
LOCAL_REPO = Path('/content/GeoDistill_RT')
DATA_ROOT = Path('/content/geolift_data')
RUN_LOCAL = Path('/content/geolift_final_run')
RUN_DRIVE = Path('/content/drive/MyDrive/GeoLift_RT_runs/v2_1_geometry_fused_1k')

METRIC_ID = '1ZtRVY67l3QkdgegSxDYtf6Cq_l-_BjQW'
DA_ID = '1DOtv8E_zW-pOkms2XUqro9vVfMnkqzbB'
GEOMETRY_FUSED_ID = '1gcaq8rFOOTEUBiGxF05ZBzomi1n2AX1q'
SUBSET_COUNT, VAL_COUNT, SEED = 1000, 200, 42
SELECTION_STRATEGY = 'min_drives'
EPOCHS, BATCH_SIZE, NUM_WORKERS = 30, 2, 2
assert SUBSET_COUNT == 1000 and VAL_COUNT == 200


In [ ]:
# 2) Mount Drive, copy source về local SSD, cài dependency
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, json
if LOCAL_REPO.exists(): shutil.rmtree(LOCAL_REPO)
if REPO_GIT_URL:
    subprocess.run(['git','clone','--depth=1',REPO_GIT_URL,str(LOCAL_REPO)], check=True)
else:
    assert DRIVE_REPO_DIR.is_dir(), DRIVE_REPO_DIR
    subprocess.run(['rsync','-a','--exclude=.git/','--exclude=venv/','--exclude=data/',
                    '--exclude=teacher_outputs/','--exclude=student_outputs/',
                    f'{DRIVE_REPO_DIR}/',f'{LOCAL_REPO}/'], check=True)
subprocess.run(['python','-m','pip','install','-q','--upgrade',
                'gdown','pyyaml','timm','tqdm','pandas','scipy','opencv-python-headless','matplotlib'], check=True)
os.chdir(LOCAL_REPO)
import torch
assert torch.cuda.is_available(), 'Chọn GPU runtime trước'
print(torch.__version__, torch.cuda.get_device_name(0))


## Gate A — inspect trực tiếp geometry tar trước khi tải 32 GB

Cell sau chỉ stream vài member đầu rồi đóng kết nối. Contract bắt buộc là `geometry_fused/train/*.npz`, key `R_G,C_G`, cùng shape, finite và `C_G∈[0,1]`. Nó đồng thời hiển thị raw ID và canonical ID để phát hiện lỗi thứ tự camera/frame.

In [ ]:
INSPECT_ROOT = DATA_ROOT/'inspect'
DOWNLOADS = DATA_ROOT/'downloads'
TEACHER_ROOT = DATA_ROOT/'teacher_outputs'
SPLIT_ROOT = DATA_ROOT/'splits'
for p in (INSPECT_ROOT,DOWNLOADS,TEACHER_ROOT,SPLIT_ROOT,RUN_LOCAL,RUN_DRIVE): p.mkdir(parents=True,exist_ok=True)
remote_report = INSPECT_ROOT/'geometry_remote_preview.json'
subprocess.run(['python','scripts/inspect_geometry_drive_tar.py',
                '--file_id',GEOMETRY_FUSED_ID,'--samples','3','--output_json',str(remote_report)], check=True)
preview = json.loads(remote_report.read_text())
assert preview['all_inspected_samples_pass']
print(json.dumps(preview, indent=2)[:7000])


## Tách đúng ba teacher archive

Ba tar chiếm khoảng 73 GB decimal khi cùng tồn tại. Cần tối thiểu 85 GiB local SSD trống. Script index cả ba archive, canonicalize geometry filename, lấy giao ID rồi chia 800 train/200 val theo raw drive tách rời. Chỉ 1.000 NPZ/role được giải nén.

In [ ]:
# 3) Download ba tar
free_gib = shutil.disk_usage('/content').free/1024**3
assert free_gib >= 85, f'Cần >=85 GiB trống, hiện có {free_gib:.1f} GiB'
archives = {
 'metric': (METRIC_ID, DOWNLOADS/'metric_coarse_train.tar', 20),
 'da': (DA_ID, DOWNLOADS/'depth_anything_train_raw.tar', 10),
 'geometry': (GEOMETRY_FUSED_ID, DOWNLOADS/'geometry_fused_train.tar', 25),
}
for role,(file_id,path,min_gib) in archives.items():
    subprocess.run(['gdown','--fuzzy',f'https://drive.google.com/file/d/{file_id}/view','-O',str(path)], check=True)
    size = path.stat().st_size/1024**3
    assert size > min_gib, f'{role} archive tải thiếu: {size:.2f} GiB'
    print(role, f'{size:.2f} GiB')


In [ ]:
# 4) Index giao ba teacher, canonicalize geometry ID, extract 800/200
subset_manifest = SPLIT_ROOT/'teacher_subset_manifest.json'
subprocess.run(['python','scripts/extract_geolift_teachers.py',
 '--metric_tar',str(archives['metric'][1]),'--da_tar',str(archives['da'][1]),
 '--geometry_tar',str(archives['geometry'][1]),'--teacher_root',str(TEACHER_ROOT),
 '--manifest',str(subset_manifest),'--count',str(SUBSET_COUNT),'--val_count',str(VAL_COUNT),
 '--seed',str(SEED),'--strategy',SELECTION_STRATEGY], check=True)
manifest_data = json.loads(subset_manifest.read_text())
assert manifest_data['train_count']==800 and manifest_data['val_count']==200
assert manifest_data['drive_overlap_train_val']==0
print({k:v for k,v in manifest_data.items() if k not in ('train_ids','val_ids','drives')})
for _,path,_ in archives.values(): path.unlink()
print('Free after selective teacher extraction:', shutil.disk_usage('/content').free/1024**3, 'GiB')


## Lấy đúng KITTI frame và tạo split có path

Teacher NPZ không chứa RGB/sparse/GT/K. Cell dưới tải ba ZIP KITTI chính thức, chỉ trích frame tương ứng với manifest, tải raw RGB/calibration cần thiết và tạo `train_800.txt`, `val_200.txt`, `test_1000.txt`.

In [ ]:
# 5) Official KITTI subset + official anonymous test 1000
KITTI_URL='https://s3.eu-central-1.amazonaws.com/avg-kitti'
official={
 'annotated':('data_depth_annotated.zip',f'{KITTI_URL}/data_depth_annotated.zip'),
 'velodyne':('data_depth_velodyne.zip',f'{KITTI_URL}/data_depth_velodyne.zip'),
 'selection':('data_depth_selection.zip',f'{KITTI_URL}/data_depth_selection.zip')}
for name,url in official.values(): subprocess.run(['wget','-q','-c',url,'-O',str(DOWNLOADS/name)],check=True)
KITTI_ROOT=DATA_ROOT/'kitti'
subprocess.run(['python','scripts/extract_official_kitti_subset.py','--manifest',str(subset_manifest),
 '--annotated_zip',str(DOWNLOADS/official['annotated'][0]),
 '--velodyne_zip',str(DOWNLOADS/official['velodyne'][0]),
 '--selection_zip',str(DOWNLOADS/official['selection'][0]),
 '--data_root',str(KITTI_ROOT),'--split_root',str(SPLIT_ROOT),'--download_dir',str(DOWNLOADS)],check=True)
for name,_ in official.values(): (DOWNLOADS/name).unlink(missing_ok=True)
for name,expected in [('train_800.txt',800),('val_200.txt',200),('test_1000.txt',1000)]:
    count=sum(bool(line.strip()) for line in (SPLIT_ROOT/name).read_text().splitlines())
    assert count==expected,(name,count)
print('KITTI splits OK')


## Gate B — inspect train và val sau khi tách

Không train nếu một role thiếu file, ID/drive bị overlap, key sai, geometry không finite hoặc confidence vượt `[0,1]`. Sau report định lượng, cell kế tiếp hiển thị trực quan riêng một mẫu train và một mẫu val.

In [ ]:
# 6a) Coverage/schema/statistics gate
subset_report=INSPECT_ROOT/'teacher_train_val_report.json'
subprocess.run(['python','scripts/inspect_teacher_subset.py','--teacher_root',str(TEACHER_ROOT),
 '--split_root',str(SPLIT_ROOT),'--samples_per_split','16','--min_coverage','1.0',
 '--output_json',str(subset_report)],check=True)
report=json.loads(subset_report.read_text())
assert report['contract_ok'] and not report['id_overlap'] and not report['drive_overlap']
print(json.dumps(report,indent=2)[:12000])


In [ ]:
# 6b) Loader thật + visualization train/val
import matplotlib.pyplot as plt, numpy as np
from src.dataset import KITTIDepthCompletionDataset
def inspect_split(split,split_file):
    ds=KITTIDepthCompletionDataset(data_root=KITTI_ROOT,split_root=SPLIT_ROOT,split_file=split_file,
        split_name=split,image_size=(352,1216),teacher_root=TEACHER_ROOT,load_teacher=True,
        load_geometry=True,geometry_fallback=False,load_mono=True,return_tensors=True)
    x=ds[0]
    assert float(x['C_G'].sum())>0 and torch.isfinite(x['R_G']).all()
    panels=[('RGB',x['rgb'].permute(1,2,0).numpy()),('Sparse',x['sparse'][0].numpy()),
            ('GT',x['gt'][0].numpy()),('Metric D_cm',x['D_cm'][0].numpy()),
            ('DA raw',x['D_da_raw'][0].numpy()),('Fused R_G',x['R_G'][0].numpy()),
            ('Fused C_G',x['C_G'][0].numpy())]
    fig,axs=plt.subplots(2,4,figsize=(22,7)); axs=axs.ravel()
    for ax,(title,img) in zip(axs,panels): ax.imshow(img,cmap=None if title=='RGB' else 'turbo'); ax.set_title(title); ax.axis('off')
    axs[-1].axis('off'); fig.suptitle(f'{split}: {x["sample_id"]}'); plt.show()
    print(split,{k:tuple(x[k].shape) for k in ('rgb','sparse','gt','D_cm','D_da_raw','R_G','C_G')},
          'R_G range=',(float(x['R_G'].min()),float(x['R_G'].max())),
          'C_G range=',(float(x['C_G'].min()),float(x['C_G'].max())))
    return ds
train_ds=inspect_split('train','train_800.txt'); val_ds=inspect_split('val','val_200.txt')


In [ ]:
# 7) Final config: geometry_fused bắt buộc, DA fallback tắt
import yaml, hashlib
paths_file=DATA_ROOT/'paths_geolift_final.yaml'
paths={'data_root':str(KITTI_ROOT),'split_root':str(SPLIT_ROOT),
 'train_split':'train_800.txt','val_split':'val_200.txt','test_split':'test_1000.txt',
 'teacher_root':str(TEACHER_ROOT),'student_root':str(RUN_LOCAL)}
paths_file.write_text(yaml.safe_dump(paths,sort_keys=False))
cfg=yaml.safe_load((LOCAL_REPO/'configs/geolift_s2_v2_1.yaml').read_text())
cfg['paths_file']=str(paths_file); cfg['train']['epochs']=EPOCHS; cfg['train']['batch_size']=BATCH_SIZE
cfg['data']['num_workers']=NUM_WORKERS; cfg['outputs']['backup_root']=str(RUN_DRIVE)
cfg['loss']['geometry_fallback']=False; cfg['teacher_checks']['require_geometry_fused']=True
cfg['mono_ssi']['enabled']=False
last_ckpt=RUN_DRIVE/'checkpoints/last.pth'
if last_ckpt.exists():
    (RUN_LOCAL/'checkpoints').mkdir(parents=True,exist_ok=True); shutil.copy2(last_ckpt,RUN_LOCAL/'checkpoints/last.pth')
    cfg['train']['resume']=str(RUN_LOCAL/'checkpoints/last.pth')
resolved_cfg=DATA_ROOT/'geolift_s2_geometry_final.yaml'; resolved_cfg.write_text(yaml.safe_dump(cfg,sort_keys=False))
for source,name in [(resolved_cfg,'resolved_config.yaml'),(paths_file,'resolved_paths.yaml'),
                    (subset_manifest,'teacher_subset_manifest.json'),(remote_report,'geometry_remote_preview.json'),
                    (subset_report,'teacher_train_val_report.json')]: shutil.copy2(source,RUN_LOCAL/name)
run_manifest={'architecture':'GeoLift-S2-v2.1','teacher_profile':'metric + fused R_G/C_G',
 'geometry_fused_drive_id':GEOMETRY_FUSED_ID,'train':800,'val':200,'test_anonymous':1000,
 'seed':SEED,'geometry_fallback':False,'slope_supervision':False,'torch':torch.__version__,
 'gpu':torch.cuda.get_device_name(0),'config_sha256':hashlib.sha256(resolved_cfg.read_bytes()).hexdigest()}
(RUN_LOCAL/'run_manifest.json').write_text(json.dumps(run_manifest,indent=2))
print(resolved_cfg.read_text())


In [ ]:
# 8) Unit tests + strict preflight + one real forward/loss
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],check=True)
from src.utils import load_project_config
from src.train_student import make_loader,preflight_teacher_coverage
from src.model_factory import build_student
from src.losses import geort_loss
import logging
loaded_cfg,loaded_paths=load_project_config(resolved_cfg)
loader=make_loader(loaded_cfg,loaded_paths,'train',training=True)
preflight_teacher_coverage(loaded_cfg,loader.dataset,logging.getLogger('strict-preflight'))
batch=next(iter(loader)); model=build_student(loaded_cfg).cuda().train()
gpu_batch={k:(v[:1].cuda() if torch.is_tensor(v) else v) for k,v in batch.items()}
with torch.autocast('cuda',dtype=torch.float16):
    pred=model(*(gpu_batch[k] for k in ('rgb','sparse','mask','ray','uv','K')))
    loss,items=geort_loss(pred,gpu_batch,loaded_cfg['loss'],loaded_cfg['schedule'],
                          loaded_cfg['schedule']['add_geometry_epoch'],loaded_cfg['mono_ssi'])
assert torch.isfinite(loss) and items['geom_valid_ratio']>0
assert float(((pred['D_full']-gpu_batch['sparse'])*gpu_batch['mask']).abs().max())==0.0
print('strict smoke loss=',float(loss.detach()),'geometry ratio=',items['geom_valid_ratio'])
del model,pred,batch,gpu_batch; torch.cuda.empty_cache()


In [ ]:
# 9) Final train/resume; last/best/log được backup Drive sau mỗi epoch
subprocess.run(['python','-m','src.train_student','--config',str(resolved_cfg)],check=True)
subprocess.run(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],check=True)
best_ckpt=RUN_LOCAL/'checkpoints/best.pth'; assert best_ckpt.exists()


In [ ]:
# 10) Val metrics từng stage + official test 1000 + component runtime
for split in ('val','test'):
    subprocess.run(['python','-m','src.infer_student','--config',str(resolved_cfg),
                    '--checkpoint',str(best_ckpt),'--split',split],check=True)
metric_file=RUN_LOCAL/'logs/infer_val_metrics_global.json'; print(metric_file.read_text())
test_png=list((RUN_LOCAL/'test_predictions/benchmark_png').glob('*.png')); assert len(test_png)==1000
shutil.make_archive(str(RUN_LOCAL/'kitti_test_1000'),'zip',RUN_LOCAL/'test_predictions/benchmark_png')
profile_file=RUN_LOCAL/'logs/geolift_component_profile.json'
subprocess.run(['python','scripts/profile_geolift_s2.py','--config',str(resolved_cfg),
                '--warmup','30','--runs','100','--output',str(profile_file)],check=True)
subprocess.run(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],check=True)
print('Saved to',RUN_DRIVE)


## Cách diễn giải kết quả

- `teacher_train_val_report.json`: bằng chứng train/val đã tách đúng, coverage ba role và thống kê `R_G/C_G`.
- `infer_val_metrics_global.json`: RMSE/MAE/iRMSE/iMAE và RMSE `D_init,D16,D8,D4,D2,D1,D_full`.
- `geolift_component_profile.json`: bottleneck runtime thực trên GPU.
- `kitti_test_1000.zip`: prediction anonymous test; không có GT local.

`geometry_fused_train.tar` chỉ chứa `R_G,C_G`. Nó đủ cho fused relative/ordinal/boundary geometry supervision nhưng không chứa `a_T,b_T,eta_T`; vì vậy run manifest ghi `slope_supervision=false`, không gọi kết quả là full DSINE slope T3.